In [3]:
import os, pandas as pd
from pathlib import Path

# Archivos individuales
NB_DIR = Path().resolve()
ROOT_DIR = (NB_DIR / ".." / "..").resolve()
test_path  = ROOT_DIR  / "datos" / "Pre-processed" / "data_finetuning_test.csv"
test_df  = pd.read_csv(test_path)

print(f"Test set: {len(test_df)} muestras")

print("Columnas:", test_df.columns.tolist())
display(test_df.head(2))

Test set: 380 muestras
Columnas: ['name', 'article', 'summary']


,name,article,summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "google/gemma-3-1b-pt"

# 1) Cargar tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

# DeepSeek Coder es un modelo estilo Code Llama / Llama-like.
# Si el tokenizer no tiene pad_token, lo igualamos al eos_token:
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2) Seleccionar dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"

# 3) Cargar el modelo en el dispositivo
#    - usa torch.float16 en GPU para ahorrar VRAM
#    - si solo tienes CPU, puedes usar float32 (por defecto) o bfloat16 si tu CPU lo soporta
dtype = torch.float16 if device == "cuda" else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map=None,   # luego movemos el modelo "a mano"
)

model.to(device)
model.eval()

print(f"Modelo cargado en: {device}, dtype={dtype}, n_params ~ {sum(p.numel() for p in model.parameters())}")

C:\Users\jsoa\AppData\Roaming\Python\Python312\site-packages\triton\windows_utils.py:404: UserWarning: Failed to find CUDA.
  warnings.warn("Failed to find CUDA.")
C:\Users\jsoa\AppData\Roaming\Python\Python312\site-packages\triton\knobs.py:212: UserWarning: Failed to find cuobjdump.exe
  warnings.warn(f"Failed to find {binary}")
C:\Users\jsoa\AppData\Roaming\Python\Python312\site-packages\triton\knobs.py:212: UserWarning: Failed to find nvdisasm.exe
  warnings.warn(f"Failed to find {binary}")
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py:11: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.5)
  from scipy.sparse import csr_matrix, issparse

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [ ]:
# ============================
# Generar PLS con DeepSeek (HF) usando columnas: name, article, summary
# ============================

import re
import time  # NEW
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

@torch.inference_mode()
def generate_pls_batch(
    data: str,
    prompt_fn,
    batch_size: int = 2,
    max_new_tokens: int = 380,     # suficiente para 4–6 oraciones
    temperature: float = 0.0,      # determinista → mejor factualidad
    top_p: float = 1.0,
    num_beams: int = 1,
    repetition_penalty: float = 1.02,
    no_repeat_ngram_size: int = 4,
):
    texts = data['article'].fillna("").astype(str).tolist()
    df_out = data.copy()

    # Calcula un input máximo seguro
    max_ctx = getattr(model.config, "max_position_embeddings", 4096)
    max_input_len = max(8, max_ctx - max_new_tokens)

    outputs = []
    latencies = []  # NEW
    pbar = tqdm(total=len(texts), desc='Generando resúmenes', unit="sample")

    try:
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            prompts = [prompt_fn(t) for t in batch_texts]

            enc = tokenizer(
                prompts,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_input_len,
            )
            input_ids = enc["input_ids"].to(model.device)
            attention_mask = enc["attention_mask"].to(model.device)

            # --- NEW: medir tiempo del batch con sync de GPU
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.perf_counter()

            gen_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=(temperature > 0.0 or top_p < 1.0),
                temperature=temperature,
                top_p=top_p,
                num_beams=num_beams,
                no_repeat_ngram_size=no_repeat_ngram_size,
                repetition_penalty=repetition_penalty,
                use_cache=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            per_item_latency = (t1 - t0)  # NEW

            # Cortar por muestra usando la longitud REAL (no el ancho padded)
            batch_out = []
            input_lens = attention_mask.sum(dim=1)  # [B]
            for row in range(input_ids.size(0)):
                ilen = int(input_lens[row].item())
                gen_only = gen_ids[row, ilen:]        # ← clave: corta desde fin del prompt de ESA fila
                text = tokenizer.decode(gen_only, skip_special_tokens=True).strip()
                batch_out.append(text)

            outputs.extend(batch_out)
            latencies.extend([per_item_latency] * len(batch_texts))  # NEW
            pbar.update(len(batch_texts))
    finally:
        pbar.close()

    df_out['gen_summary'] = outputs
    df_out['latency_s'] = latencies  # NEW
    return df_out

# O el CoT factual corto
def generar_prompt(scientific_text: str) -> str:
    return (
        "Usingthefollowingabstractofabiomedicalstudyasinput,generateaPlainLanguageSummary\n"
        "(PLS)understandablebyanypatient,regardlessoftheirhealthliteracy.Ensurethatthegeneratedtext\n"
        "adherestothefollowinginstructionswhichshouldbefollowedstep-by-step:\n"
        "a.SpecificStructure:ThegeneratedPLSshouldbepresentedinalogicalorder,usingthefollowing\n"
        "order:\n"
        "1. PlainTitle\n"
        "2. Rationale\n"
        "3. TrialDesign\n"
        "4. Results\n"
        "b.Sectionsshouldbeauthoredfollowingtheseparameters:\n"
        "1. PlainTitle:Simplifiedtitleunderstandabletoalaypersonthatsummarizestheresearchthatwas\n"
        "done.\n"
        "2. Rationale: Include: backgroundor studyrationaleprovidingageneraldescriptionof the\n"
        "condition,whatitmaycauseorwhyitisaburdenforthepatients;thereasonandmainhypothesis\n"
        "forthestudy;andwhythestudyisneeded,andwhythestudymedicationhasthepotentialto\n"
        "treatthecondition.\n"
        "3. TrialDesign:Answer‘Howisthisstudydesigned?’ Includethedescriptionof thedesign,\n"
        "descriptionofstudyandpatientpopulation(age,healthcondition,gender),andtheexpected\n"
        "amountoftimeapersonwillbeinthestudy.\n"
        "4. Results:Answer‘Whatwerethemainresultsofthestudy’,includethebenefitsforthepatients,\n"
        "howthestudywasrelevantfortheareaofstudy,andtheconclusionsfromtheinvestigator.\n"
        "c.ConsistencyandReplicability:ThegeneratedPLSshouldbeconsistentregardlessoftheorderof\n"
        "sentencesorthespecificphrasingusedintheinputprotocoltext.\n"
        "d.CompliancewithPlainLanguageGuidelines:ThegeneratedPLSmustfollowalltheseplain\n"
        "languageguidelines:\n"
        "• Havereadabilitygradelevelof6orbelow.\n"
        "• Donothavejargon.Alltechnicalormedicalwordsortermsshouldbedefinedorbrokendown\n"
        "intosimpleandlogicalexplanations.\n"
        "•Activevoice,notpassive.\n"
        "•Mostlyoneortwosyllablewords.\n"
        "• Sentencesof15wordsorless.\n"
        "• Shortparagraphsof3-5sentences.\n"
        "• Simplenumbers(e.g.,ratios,nopercentages).\n"
        "e.DonotinventContent:TheAImodelshouldnotinventinformation. IftheAImodelincludesdata\n"
        "otherthantheonegivenintheinputabstract,theAImodelshouldguaranteesuchdataisverifiedand\n"
        "real.\n"
        "f.AimforanapproximatePLSlengthof500-900words.\n\n"
        f"Cientific Text: {scientific_text}\n\n"
        "PLS Text:"
    )

DATA_DIR = Path(ROOT_DIR) / "datos" / "pre-processed"
RESULTS_DIR = Path(ROOT_DIR) / "resultados" / "resumenes"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df_test = pd.read_csv(DATA_DIR / "data_finetuning_test.csv")
df_output = generate_pls_batch(
    data=df_test,
    prompt_fn=generar_prompt,   # o prompt_sencillo
    batch_size=2,
    max_new_tokens=380,
    temperature=0.0,
    top_p=1.0,
)

csv_out = RESULTS_DIR / "summaries_deepseek3.csv"

df_output.to_csv(csv_out, index=False, encoding="utf-8")
print(f"Guardado CSV con PLS: {csv_out}")
print("Latencia promedio (s):", df_output["latency_s"].mean())
#3.7 gb a 6 GB VRAM

Generando resúmenes: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 380/380 [51:26<00:00,  8.12s/sample]

Guardado CSV con PLS: summaries_deepseek.csv
Latencia promedio (s): 16.23739412110942
